In [6]:
import numpy as np
import os

# CHANGE THIS to the path of the folder shown in your screenshot
scene_path = "data/scannet_data/val/scene0011_00" 

# List of files to check
files_to_check = [
    "coord.npy", 
    "color.npy", 
    "normal.npy", 
    "instance.npy", 
    "segment20.npy",
    "segment200.npy"
]

print(f"--- Inspecting scene: {scene_path} ---\n")

for filename in files_to_check:
    file_path = os.path.join(scene_path, filename)
    
    if os.path.exists(file_path):
        # Load the .npy file
        data = np.load(file_path)
        
        print(f"📄 File: {filename}")
        print(f"   Shape: {data.shape}")
        print(f"   Type:  {data.dtype}")
        
        # Show min/max to understand value ranges (useful for color/coords)
        if data.size > 0:
            print(f"   Min:   {np.min(data)}")
            print(f"   Max:   {np.max(data)}")
            
            # Print first 2 entries to see what the data actually looks like
            print(f"   Sample data:\n{data[:2]}")
        
        print("-" * 30)
    else:
        print(f"❌ File not found: {filename}")

--- Inspecting scene: data/scannet_data/val/scene0011_00 ---

📄 File: coord.npy
   Shape: (237360, 3)
   Type:  float32
   Min:   -0.032526493072509766
   Max:   8.213502883911133
   Sample data:
[[2.5091114  0.4083811  0.14877559]
 [2.5156426  0.4059527  0.14168811]]
------------------------------
📄 File: color.npy
   Shape: (237360, 3)
   Type:  uint8
   Min:   2
   Max:   255
   Sample data:
[[35 33 38]
 [34 32 39]]
------------------------------
📄 File: normal.npy
   Shape: (237360, 3)
   Type:  float32
   Min:   -0.9999940991401672
   Max:   0.9999979734420776
   Sample data:
[[0.19109516 0.92176497 0.3371149 ]
 [0.35799062 0.9311391  0.06885417]]
------------------------------
📄 File: instance.npy
   Shape: (237360,)
   Type:  int64
   Min:   -1
   Max:   32
   Sample data:
[27 27]
------------------------------
📄 File: segment20.npy
   Shape: (237360,)
   Type:  int64
   Min:   -1
   Max:   19
   Sample data:
[14 14]
------------------------------
📄 File: segment200.npy
   Shape

In [1]:
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset

class ScanNetDataset(Dataset):
    def __init__(self, data_root, transform=None):
        """
        Args:
            data_root (str): Path to the folder containing scene folders 
                             (e.g., 'data/scannet_processed/val').
            transform (callable, optional): Sonata transform pipeline.
        """
        self.data_root = data_root
        self.transform = transform
        
        # specific to your directory structure: data_root/sceneXXXX_XX/*.npy
        # We search for all folders inside data_root
        self.scene_paths = sorted(glob.glob(os.path.join(data_root, "scene*")))
        
        if len(self.scene_paths) == 0:
            raise ValueError(f"No scene folders found in {data_root}. Check your path.")

        print(f"Loaded {len(self.scene_paths)} scenes from {data_root}")

    def __len__(self):
        return len(self.scene_paths)

    def __getitem__(self, idx):
        scene_path = self.scene_paths[idx]
        scene_name = os.path.basename(scene_path)

        # Load the specific .npy files you identified
        try:
            coord = np.load(os.path.join(scene_path, "coord.npy")).astype(np.float32)
            color = np.load(os.path.join(scene_path, "color.npy")).astype(np.float32)
            normal = np.load(os.path.join(scene_path, "normal.npy")).astype(np.float32)
            
            # Load labels if they exist (usually for train/val)
            segment_path = os.path.join(scene_path, "segment20.npy")
            instance_path = os.path.join(scene_path, "instance.npy")
            
            if os.path.exists(segment_path):
                segment = np.load(segment_path).astype(np.int64)
            else:
                segment = np.zeros(coord.shape[0], dtype=np.int64) - 1 # Ignore index

            if os.path.exists(instance_path):
                instance = np.load(instance_path).astype(np.int64)
            else:
                instance = np.zeros(coord.shape[0], dtype=np.int64) - 1

        except FileNotFoundError as e:
            raise FileNotFoundError(f"Missing required .npy file in {scene_name}: {e}")

        # Construct the dictionary expected by Sonata/Pointcept
        data_dict = {
            "coord": coord,
            "color": color,
            "normal": normal,
            "segment20": segment,  
            "instance": instance,
            "name": scene_name,
            "id": idx
        }

        # Apply Sonata transforms (grid sampling, normalization, etc.)
        if self.transform:
            data_dict = self.transform(data_dict)

        return data_dict

In [2]:
train_path = 'data/scannet_data/train'
dataset = ScanNetDataset(data_root=train_path)
print(f"Dataset length: {len(dataset)}")

Loaded 1201 scenes from data/scannet_data/train
Dataset length: 1201


In [3]:
data_item = dataset[0]
print(f"Data item keys: {list(data_item.keys())}")
print(f"Coordinates shape: {data_item['coord'].shape}, sample value {data_item['coord'][:2]}")
print(f"Color shape: {data_item['color'].shape}, sample value {data_item['color'][:2]}")
print(f"Normal shape: {data_item['normal'].shape}, sample value {data_item['normal'][:2]}")
print(f"Segment shape: {data_item['segment20'].shape}, sample value {data_item['segment20'][:2]}")
print(f"Instance shape: {data_item['instance'].shape}, sample value {data_item['instance'][:2]}")

Data item keys: ['coord', 'color', 'normal', 'segment20', 'instance', 'name', 'id']
Coordinates shape: (81369, 3), sample value [[0.5324214  4.5172734  0.26304942]
 [0.53404164 4.552089   0.262302  ]]
Color shape: (81369, 3), sample value [[101. 107.  90.]
 [ 88.  83.  78.]]
Normal shape: (81369, 3), sample value [[ 0.8616817  -0.02587385 -0.5067754 ]
 [ 0.9884297   0.14236335 -0.05226134]]
Segment shape: (81369,), sample value [13 13]
Instance shape: (81369,), sample value [5 5]


In [4]:


import open3d as o3d
import sonata
import torch

try:
    import flash_attn
except ImportError:
    flash_attn = None


def get_pca_color(feat, brightness=1.25, center=True):
    u, s, v = torch.pca_lowrank(feat, center=center, q=6, niter=5)
    projection = feat @ v
    projection = projection[:, :3] * 0.6 + projection[:, 3:6] * 0.4
    min_val = projection.min(dim=-2, keepdim=True)[0]
    max_val = projection.max(dim=-2, keepdim=True)[0]
    div = torch.clamp(max_val - min_val, min=1e-6)
    color = (projection - min_val) / div * brightness
    color = color.clamp(0.0, 1.0)
    return color


if __name__ == "__main__":
    # set random seed
    # (random seed affect pca color, yet change random seed need manual adjustment kmeans)
    # (the pca prevent in paper is with another version of cuda and pytorch environment)
    sonata.utils.set_seed(53124)
    # Load model
    if flash_attn is not None:
        model = sonata.load("sonata", repo_id="facebook/sonata").cuda()
    else:
        custom_config = dict(
            enc_patch_size=[1024 for _ in range(5)],  # reduce patch size if necessary
            enable_flash=False,
        )
        model = sonata.load(
            "sonata", repo_id="facebook/sonata", custom_config=custom_config
        ).cuda()
    # Load default data transform pipeline
    transform = sonata.transform.default()
    
    # Load data
    # point = sonata.data.load("sample1")
    # inspect loaded data
    point = data_item

    
    # point.pop("segment200")
    segment = point.pop("segment20")
    point["segment"] = segment  # two kinds of segment exist in ScanNet, only use one
    original_coord = point["coord"].copy()
    point = transform(point)

    with torch.inference_mode():
        for key in point.keys():
            if isinstance(point[key], torch.Tensor):
                point[key] = point[key].cuda(non_blocking=True)
        # model forward:
        point = model(point)
        # upcast point feature
        # Point is a structure contains all the information during forward
        for _ in range(2):
            assert "pooling_parent" in point.keys()
            assert "pooling_inverse" in point.keys()
            parent = point.pop("pooling_parent")
            inverse = point.pop("pooling_inverse")
            parent.feat = torch.cat([parent.feat, point.feat[inverse]], dim=-1)
            point = parent
        while "pooling_parent" in point.keys():
            assert "pooling_inverse" in point.keys()
            parent = point.pop("pooling_parent")
            inverse = point.pop("pooling_inverse")
            parent.feat = point.feat[inverse]
            point = parent

        # here point is down-sampled by GridSampling in default transform pipeline
        # feature of point cloud in original scale can be acquired by:
        _ = point.feat[point.inverse]

        # PCA
        pca_color = get_pca_color(point.feat, brightness=1.2, center=True)

    # inverse back to original scale before grid sampling
    # point.inverse is acquired from the GirdSampling transform
    original_pca_color = pca_color[point.inverse]
    pcd = o3d.geometry.PointCloud()
    # pcd.points = o3d.utility.Vector3dVector(original_coord)
    # pcd.colors = o3d.utility.Vector3dVector(original_pca_color.cpu().detach().numpy())
    # o3d.visualization.draw_geometries([pcd])
    # or
    o3d.visualization.draw_plotly([pcd])

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(point.coord.cpu().detach().numpy())
    pcd.colors = o3d.utility.Vector3dVector(pca_color.cpu().detach().numpy())
    o3d.io.write_point_cloud("pca.ply", pcd)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading checkpoint from HuggingFace: sonata ...
Model params: 108.46M


In [1]:
import os
import glob
import copy
import numpy as np
import torch
from torch.utils.data import Dataset
import sonata  # Assuming your custom sonata package is available

# --- CONSTANTS ---
# Standard ScanNet v2 20-class mapping
CLASS_NAMES = {
    0: "wall", 1: "floor", 2: "cabinet", 3: "bed", 4: "chair",
    5: "sofa", 6: "table", 7: "door", 8: "window", 9: "bookshelf",
    10: "picture", 11: "counter", 12: "desk", 13: "curtain",
    14: "refrigerator", 15: "shower curtain", 16: "toilet",
    17: "sink", 18: "bathtub", 19: "otherfurniture"
}

# Classes to exclude from being Ground Truth
IGNORED_CLASS_IDS = {0, 1, 19}

# Valid candidates for fallback (if a scene is empty of objects)
VALID_CLASS_IDS = [i for i in CLASS_NAMES.keys() if i not in IGNORED_CLASS_IDS]


class ScanNetTextDataset(Dataset):
    def __init__(self, data_root, transform=None):
        """
        Args:
            data_root (str): Path to the folder containing scene folders 
                             (e.g., 'data/scannet_processed/val').
            transform (callable, optional): Sonata transform pipeline.
        """
        self.data_root = data_root
        self.transform = transform
        
        # specific to your directory structure: data_root/sceneXXXX_XX/*.npy
        # We search for all folders inside data_root
        self.scene_paths = sorted(glob.glob(os.path.join(data_root, "scene*")))
        
        if len(self.scene_paths) == 0:
            raise ValueError(f"No scene folders found in {data_root}. Check your path.")

        print(f"Loaded {len(self.scene_paths)} scenes for Text-Guided Training.")

    def __len__(self):
        return len(self.scene_paths)

    def __getitem__(self, idx):
        scene_path = self.scene_paths[idx]
        scene_name = os.path.basename(scene_path)

        # 1. Load Data strictly (No try/except, no exist checks)
        # If a file is missing, this will raise FileNotFoundError immediately.
        coord = np.load(os.path.join(scene_path, "coord.npy")).astype(np.float32)
        color = np.load(os.path.join(scene_path, "color.npy")).astype(np.float32)
        normal = np.load(os.path.join(scene_path, "normal.npy")).astype(np.float32)
        segment = np.load(os.path.join(scene_path, "segment20.npy")).astype(np.int64)
        instance = np.load(os.path.join(scene_path, "instance.npy")).astype(np.int64)

        # --- Dynamic Ground Truth Selection (Text Logic) ---
        
        # Find all unique classes present in this scene based on the loaded segment
        unique_classes = np.unique(segment)
        
        # Filter out ignored classes (Wall, Floor, Other) and -1 (ignore index)
        candidates = [c for c in unique_classes if c not in IGNORED_CLASS_IDS and c != -1]

        if len(candidates) > 0:
            # Pick a random object present in the scene (Positive Sample)
            # EXPLICITLY CAST TO PYTHON INT to avoid numpy.int64 tensor conversion errors
            target_cid = int(np.random.choice(candidates))
        else:
            # SKIP MODE: Return None for scenes with no valid objects
            # The collate function will filter these out
            return None

        # Get the text label
        target_text = CLASS_NAMES[target_cid]

        # 2. Construct Data Dictionary (Geometric Data Only)
        # This dict goes into the transform pipeline. Keys here might be dropped/renamed.
        data_dict = {
            "coord": coord,
            "color": color,
            "normal": normal,
            "segment20": segment, 
            "instance": instance,
        }

        # SAVE ORIGINAL DATA BEFORE TRANSFORM (for training labels)
        # Transform will voxelize/crop/augment, potentially losing segment info
        original_coord = coord.copy()
        original_segment = segment.copy()

        # 3. Apply Sonata Transform (Voxelization, Augmentation)
        if self.transform:
            data_dict = self.transform(data_dict)

        # 4. Construct Metadata Dictionary (Protected Data)
        # These keys will bypass the transform logic entirely to prevent loss.
        meta_dict = {
            "name": scene_name,
            "id": idx,             
            "target_cid": target_cid, # For mask generation later
            "text": target_text,      # For CLIP encoding
            # ADD: Original dense data for label propagation
            "original_coord": torch.from_numpy(original_coord).float(),
            "original_segment": torch.from_numpy(original_segment).long(),
        }

        return data_dict, meta_dict

def training_collate_fn(batch):
    """
    Collate function that handles the separated data and metadata.
    Args:
        batch: List of tuples [(data_dict, meta_dict), ...] or None for skipped scenes
    Returns:
        collated_data: Sonata Point object (batched coordinates/features)
        collated_meta: List of metadata dictionaries
        Returns None if all samples in batch are invalid
    """
    # Filter out None samples (scenes with no valid objects)
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None

    # Unzip the batch into two lists
    data_batch = [item[0] for item in batch]
    meta_batch = [item[1] for item in batch]

    # Use Sonata's native collate function ONLY on the data part
    collated_data = sonata.data.collate_fn(data_batch)
    
    # Return separated tuple
    return collated_data, meta_batch


In [2]:

print(f"Dataset length: {len(dataset)}")

Testing Dataset with root: data/scannet_data/val
Loaded 312 scenes for Text-Guided Training.
Dataset length: 312


In [2]:
from torch.utils.data import DataLoader
import sonata
from sonata.scannet_text import ScanNetTextDataset, training_collate_fn
import torch

CUSTOM_SONATA_CONFIG = dict(
    enc_patch_size=[1024 for _ in range(5)],
    enable_flash=False,
    enc_mode=True,
    freeze_encoder=True, 
)

data_root = "data/scannet_data/val" # Adjust as needed
print(f"Testing Dataset with root: {data_root}")
transform = sonata.transform.default()
dataset = ScanNetTextDataset(data_root=data_root, transform=transform, max_points=100000)

loader = DataLoader(dataset, batch_size=16, 
                        collate_fn=training_collate_fn, shuffle=False)

point_model = sonata.load("sonata", repo_id="facebook/sonata", custom_config=CUSTOM_SONATA_CONFIG)
point_model = point_model.to('cuda')
point_model.eval()

next_iter = iter(loader)
batch = next(next_iter)  # may be (collated_data, collated_meta) or None
if batch is None:
    raise RuntimeError("Batch is None (all samples skipped).")
collated_data, collated_meta = batch

# Move all tensors in collated_data to CUDA
for k in list(collated_data.keys()):
    v = collated_data[k]
    if isinstance(v, torch.Tensor):
        collated_data[k] = v.cuda(non_blocking=True)
    

print(f"Collated Data Keys: {list(collated_data.keys())}")
print(f"Number of Metadata Entries: {len(collated_meta)}")

point_data = point_model(collated_data)
print(f"Point Data Keys after Model Forward: {list(point_data.keys())}")

Testing Dataset with root: data/scannet_data/val
Loaded 312 scenes for Text-Guided Training.
Loading checkpoint from HuggingFace: sonata ...
Model params: 0.00M
Model params: 0.00M
Collated Data Keys: ['coord', 'grid_coord', 'color', 'inverse', 'offset', 'feat']
Number of Metadata Entries: 16
Collated Data Keys: ['coord', 'grid_coord', 'color', 'inverse', 'offset', 'feat']
Number of Metadata Entries: 16
Point Data Keys after Model Forward: ['feat', 'coord', 'grid_coord', 'batch', 'color', 'pooling_inverse', 'pooling_parent', 'offset', 'order', 'serialized_depth', 'serialized_code', 'serialized_order', 'serialized_inverse', 'sparse_shape', 'sparse_conv_feat', 'pad', 'unpad', 'cu_seqlens_key', 'stage_embeddings', 'stage_coords', 'stage_pooling_inverses']
Point Data Keys after Model Forward: ['feat', 'coord', 'grid_coord', 'batch', 'color', 'pooling_inverse', 'pooling_parent', 'offset', 'order', 'serialized_depth', 'serialized_code', 'serialized_order', 'serialized_inverse', 'sparse_shape

In [3]:
for t in point_data['stage_embeddings']:
    print(f"Stage Embedding Shape: {t.shape}")

Stage Embedding Shape: torch.Size([1314329, 48])
Stage Embedding Shape: torch.Size([680187, 96])
Stage Embedding Shape: torch.Size([213885, 192])
Stage Embedding Shape: torch.Size([56437, 384])
Stage Embedding Shape: torch.Size([13913, 512])


In [ ]:

import sonata
from sonata.scannet_text import ScanNetTextDataset
import torch

CUSTOM_SONATA_CONFIG = dict(
    enc_patch_size=[1024 for _ in range(5)],
    enable_flash=False,
    enc_mode=True,
    freeze_encoder=True, 
)

data_root = "data/scannet_data/val" # Adjust as needed
print(f"Testing Dataset with root: {data_root}")
transform = sonata.transform.default()
dataset = ScanNetTextDataset(data_root=data_root, transform=transform)


point_model = sonata.load("sonata", repo_id="facebook/sonata", custom_config=CUSTOM_SONATA_CONFIG)
point_model = point_model.to('cuda')
point_model.eval()

dataset[0]

Testing Dataset with root: data/scannet_data/val
Loaded 312 scenes for Text-Guided Training.
Loading checkpoint from HuggingFace: sonata ...
Model params: 0.00M
Model params: 0.00M


({'coord': tensor([[ 2.8009, -2.7214,  0.8724],
          [ 2.8021, -2.4417,  0.6706],
          [ 2.8027, -2.4446,  0.6893],
          ...,
          [-2.2867,  1.6211,  1.5823],
          [-2.2863,  1.6221,  1.5562],
          [-2.2866,  1.6244,  1.5324]]),
  'grid_coord': tensor([[287,  70,  43],
          [287,  84,  33],
          [287,  84,  34],
          ...,
          [ 32, 288,  79],
          [ 32, 288,  77],
          [ 32, 288,  76]]),
  'color': tensor([[0.1333, 0.1059, 0.0824],
          [0.3059, 0.1843, 0.1686],
          [0.2431, 0.1451, 0.1333],
          ...,
          [0.9255, 0.9255, 0.9255],
          [0.8745, 0.8745, 0.8745],
          [0.9647, 0.9647, 0.9647]]),
  'inverse': tensor([155118, 155118, 155119,  ...,   5863,  22958,  22959]),
  'offset': tensor([164772]),
  'feat': tensor([[ 2.8009e+00, -2.7214e+00,  8.7240e-01,  ..., -8.7836e-02,
           -4.1993e-01,  9.0324e-01],
          [ 2.8021e+00, -2.4417e+00,  6.7063e-01,  ..., -4.3957e-01,
           -6.

In [25]:
import os

# Make cache directory
cache_dir = "outputs/cache"
os.makedirs(cache_dir, exist_ok=True)

print("Starting caching process...")

# Iterate over first 5 samples
for idx in range(min(5, len(dataset))):
    # Handle None samples (scenes with no valid objects)
    data = dataset[idx]
    if data is None:
        print(f"Sample {idx}: Skipped (no valid objects)")
        continue
    
    point_data, meta_data = data
    
    # Move point_data tensors to CUDA
    for k in list(point_data.keys()):
        v = point_data[k]
        if isinstance(v, torch.Tensor):
            point_data[k] = v.cuda(non_blocking=True)
    
    # Forward pass through model
    with torch.no_grad():
        point_feat = point_model(point_data)
    
    # Create cache dictionary
    cache_path = os.path.join(cache_dir, f"{meta_data['name']}_cache.pth")
    
    # Save as PyTorch file (better than numpy for complex structures)
    data_dict = {
        "stage_embeddings": point_feat['stage_embeddings'],
        "stage_coords": point_feat['stage_coords'],
        "stage_inverse": point_feat['stage_pooling_inverses'],
        "meta_data": meta_data
    }
    
    torch.save(data_dict, cache_path)
    print(f"Sample {idx}: Cached {meta_data['name']} -> {cache_path}")

print(f"\nCaching complete! Saved to {cache_dir}")


Starting caching process...
Sample 0: Cached scene0011_00 -> outputs/cache/scene0011_00_cache.pth
Sample 0: Cached scene0011_00 -> outputs/cache/scene0011_00_cache.pth
Sample 1: Cached scene0011_01 -> outputs/cache/scene0011_01_cache.pth
Sample 1: Cached scene0011_01 -> outputs/cache/scene0011_01_cache.pth
Sample 2: Cached scene0015_00 -> outputs/cache/scene0015_00_cache.pth
Sample 3: Cached scene0019_00 -> outputs/cache/scene0019_00_cache.pth
Sample 2: Cached scene0015_00 -> outputs/cache/scene0015_00_cache.pth
Sample 3: Cached scene0019_00 -> outputs/cache/scene0019_00_cache.pth
Sample 4: Cached scene0019_01 -> outputs/cache/scene0019_01_cache.pth

Caching complete! Saved to outputs/cache
Sample 4: Cached scene0019_01 -> outputs/cache/scene0019_01_cache.pth

Caching complete! Saved to outputs/cache


In [26]:
#load cached sample for inspection
import torch
sample_path = "outputs/cache/scene0011_00_cache.pth"
cached_data = torch.load(sample_path)
for emb in cached_data['stage_embeddings']:
    print(f"Stage Embedding Shape: {emb.shape}")
for coors in cached_data['stage_coords']:
    print(f"Stage Coords Shape: {coors.shape}")
for inv in cached_data['stage_inverse']:
    if inv is not None:
        print(f"Stage Inverse Shape: {inv.shape}")
        print(f"Stage Inverse Sample: {inv[:5]}")



Stage Embedding Shape: torch.Size([164772, 48])
Stage Embedding Shape: torch.Size([58935, 96])
Stage Embedding Shape: torch.Size([15692, 192])
Stage Embedding Shape: torch.Size([3871, 384])
Stage Embedding Shape: torch.Size([939, 512])
Stage Coords Shape: torch.Size([164772, 3])
Stage Coords Shape: torch.Size([58935, 3])
Stage Coords Shape: torch.Size([15692, 3])
Stage Coords Shape: torch.Size([3871, 3])
Stage Coords Shape: torch.Size([939, 3])
Stage Inverse Shape: torch.Size([164772])
Stage Inverse Sample: tensor([58864, 58866, 58867, 58867, 58867], device='cuda:0')
Stage Inverse Shape: torch.Size([58935])
Stage Inverse Sample: tensor([0, 1, 1, 2, 2], device='cuda:0')
Stage Inverse Shape: torch.Size([15692])
Stage Inverse Sample: tensor([0, 1, 1, 2, 2], device='cuda:0')
Stage Inverse Shape: torch.Size([3871])
Stage Inverse Sample: tensor([10, 11, 11, 12, 12], device='cuda:0')
